In [1]:
import os
SYNOPTIC_TOKEN = os.getenv("SYNOPTIC_TOKEN")

In [2]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import json
import time
from datetime import datetime, timezone
from typing import Dict, Any, Optional, List, Tuple

import requests
import pandas as pd
import math

# -------------------------
# CONFIG
# -------------------------
SYNOPTIC_TOKEN = os.getenv("SYNOPTIC_TOKEN", "").strip()

STATIONS_ALL_CSV = "/Users/irene/projects/out/stations_all.csv"

OUT_NETWORKS_IN_IDAHO = "/Users/irene/projects/out/networks_in_idaho.csv"
OUT_META_COMPLETENESS = "/Users/irene/projects/out/metadata_completeness_by_network.csv"
OUT_OVERLAP_ITD_RWIS   = "/Users/irene/projects/out/overlap_synoptic_vs_ITD_Rwis.csv"

# Synoptic
SYN_NETWORKS_URL = "https://api.synopticdata.com/v2/networks"  # docs  [oai_citation:3‡Synoptic Docs](https://docs.synopticdata.com/services/networks?utm_source=chatgpt.com)

# ITD RWIS (ArcGIS REST)
ITD_RWIS_LAYER = "https://gis.itd.idaho.gov/arcgisprod/rest/services/ArcGISOnline/RoadFeatureLayers/MapServer/6"  #  [oai_citation:4‡GIS Idaho](https://gis.itd.idaho.gov/arcgisprod/rest/services/ArcGISOnline/RoadFeatureLayers/MapServer/6?utm_source=chatgpt.com)
ITD_RWIS_QUERY = ITD_RWIS_LAYER + "/query"  #  [oai_citation:5‡GIS Idaho](https://gis.itd.idaho.gov/arcgisprod/rest/services/ArcGISOnline/RoadFeatureLayers/MapServer/6/query?utm_source=chatgpt.com)

# Overlap threshold (meters)
MAX_DIST_M = 200.0


def utcnow_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def clean_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip()


def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2.0)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


# -------------------------
# 1) Synoptic networks catalog
# -------------------------
def fetch_synoptic_networks(token: str, timeout: int = 60) -> pd.DataFrame:
    if not token:
        raise ValueError("SYNOPTIC_TOKEN missing.")
    r = requests.get(SYN_NETWORKS_URL, params={"token": token}, timeout=timeout)
    r.raise_for_status()
    js = r.json()
    # docs show list under MNET  [oai_citation:6‡Synoptic Docs](https://docs.synopticdata.com/services/networks?utm_source=chatgpt.com)
    df = pd.DataFrame(js.get("MNET", []))
    if df.empty:
        # fallback if format changes
        df = pd.DataFrame(js.get("NETWORKS", []))
    return df


# -------------------------
# 2) networks_in_idaho.csv
#    uses stations_all.csv (already contains Synoptic stations + network_id fields)
# -------------------------
def build_networks_in_idaho(stations_all: pd.DataFrame, networks_df: pd.DataFrame) -> pd.DataFrame:
    syn = stations_all[stations_all["source"] == "SYNOPTIC"].copy()
    # Your ingest uses 'network_id' OR 'mnet_id'. Support both.
    if "mnet_id" in syn.columns:
        syn["mnet_id"] = syn["mnet_id"]
    elif "network_id" in syn.columns:
        syn["mnet_id"] = syn["network_id"]
    else:
        syn["mnet_id"] = None

    syn["mnet_id"] = syn["mnet_id"].astype(str).str.strip()
    syn = syn[syn["mnet_id"].notna() & (syn["mnet_id"] != "") & (syn["mnet_id"] != "None")]

    counts = syn.groupby("mnet_id", dropna=True).agg(
        n_stations=("source_station_id", "nunique"),
        n_rows=("source_station_id", "size"),
    ).reset_index()

    # normalize network_df id column
    net = networks_df.copy()
    net["mnet_id"] = net["ID"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
     # commonly 'ID' in Synoptic networks payload
    if "ID" in net.columns:
        net["mnet_id"] = (
            net["ID"]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )
    elif "id" in net.columns:
        net["mnet_id"] = (
            net["id"]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )
    # bring human-readable names if present
    name_cols = [c for c in ["SHORTNAME", "LONGNAME", "URL", "PROGRAM", "CATEGORY"] if c in net.columns]
    net_small = net[["mnet_id"] + name_cols].drop_duplicates("mnet_id")

    out = counts.merge(net_small, on="mnet_id", how="left")
    out["generated_utc"] = utcnow_iso()
    out = out.sort_values(["n_stations", "mnet_id"], ascending=[False, True]).reset_index(drop=True)
    return out


# -------------------------
# 3) metadata completeness by network
#    uses your enriched station fields if present; otherwise uses what exists in stations_all.csv
# -------------------------
def metadata_completeness_by_network(stations_all: pd.DataFrame) -> pd.DataFrame:
    syn = stations_all[stations_all["source"] == "SYNOPTIC"].copy()

    # ---- robust mnet_id mapping ----
    # prefer mnet_id; fallback to network_id; fallback to MNET_ID (se presente con altro nome)
    if "mnet_id" in syn.columns:
        syn["mnet_id_norm"] = syn["mnet_id"]
    elif "network_id" in syn.columns:
        syn["mnet_id_norm"] = syn["network_id"]
    elif "MNET_ID" in syn.columns:
        syn["mnet_id_norm"] = syn["MNET_ID"]
    else:
        syn["mnet_id_norm"] = None

    syn["mnet_id_norm"] = syn["mnet_id_norm"].astype(str).str.strip()
    syn = syn[syn["mnet_id_norm"].notna() & (syn["mnet_id_norm"] != "") & (syn["mnet_id_norm"] != "None")]

    # ---- if empty, return empty but well-formed df ----
    cols = [
        "mnet_id", "n_stations",
        "has_latlon_pct", "has_elevation_pct", "has_timezone_pct", "has_county_pct",
        "has_providers_pct", "has_siting_pct", "has_period_of_record_pct",
        "generated_utc"
    ]
    if syn.empty:
        return pd.DataFrame(columns=cols)

    def col_exists(c: str) -> bool:
        return c in syn.columns

    fields = {
        "has_latlon": lambda d: d["latitude"].notna() & d["longitude"].notna(),
        "has_elevation": lambda d: d["elevation_m"].notna() if "elevation_m" in d.columns else pd.Series(False, index=d.index),
        "has_timezone": lambda d: d["timezone"].notna() if col_exists("timezone") else pd.Series(False, index=d.index),
        "has_county": lambda d: d["county"].notna() if col_exists("county") else pd.Series(False, index=d.index),
        "has_providers": lambda d: d["providers_json"].notna() if col_exists("providers_json") else pd.Series(False, index=d.index),
        "has_siting": lambda d: d["siting_json"].notna() if col_exists("siting_json") else pd.Series(False, index=d.index),
        "has_period_of_record": lambda d: d["period_of_record_json"].notna() if col_exists("period_of_record_json") else pd.Series(False, index=d.index),
    }

    rows = []
    for mnet_id, g in syn.groupby("mnet_id_norm", dropna=True):
        n = len(g)
        rec = {"mnet_id": mnet_id, "n_stations": int(g["source_station_id"].nunique() if "source_station_id" in g.columns else n)}
        for k, fn in fields.items():
            frac = float(fn(g).mean()) if n > 0 else 0.0
            rec[k + "_pct"] = round(100.0 * frac, 2)
        rows.append(rec)

    out = pd.DataFrame(rows)
    out = out.sort_values("n_stations", ascending=False).reset_index(drop=True)
    out["generated_utc"] = utcnow_iso()
    return out


# -------------------------
# 4) ITD RWIS download and overlap with Synoptic
# -------------------------
def fetch_itd_rwis_points(timeout: int = 60) -> pd.DataFrame:
    """
    Downloads RWIS point features from ITD ArcGIS REST.
    Uses returnGeometry + outFields=* and f=geojson (supported)  [oai_citation:7‡GIS Idaho](https://gis.itd.idaho.gov/arcgisprod/rest/services/ArcGISOnline/RoadFeatureLayers/MapServer/6?utm_source=chatgpt.com)
    """
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "f": "geojson",
        "resultRecordCount": 10000,
    }
    r = requests.get(ITD_RWIS_QUERY, params=params, timeout=timeout)
    r.raise_for_status()
    js = r.json()

    feats = js.get("features", [])
    rows = []
    for f in feats:
        geom = f.get("geometry") or {}
        props = f.get("properties") or {}
        lon, lat = None, None
        # geojson point: coordinates [lon, lat]
        coords = geom.get("coordinates")
        if isinstance(coords, list) and len(coords) >= 2:
            lon, lat = coords[0], coords[1]
        rows.append({
            "source": "ITD_RWIS",
            "station_name": props.get("SITE") or props.get("site") or props.get("NAME") or props.get("Name"),
            "itd_site_id": props.get("SITEID") or props.get("siteid") or props.get("OBJECTID") or props.get("objectid"),
            "latitude": pd.to_numeric(lat, errors="coerce"),
            "longitude": pd.to_numeric(lon, errors="coerce"),
            "raw_props_json": json.dumps(props, ensure_ascii=False),
        })

    df = pd.DataFrame(rows).dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    return df


def overlap_synoptic_vs_itd(stations_all: pd.DataFrame, itd_df: pd.DataFrame, max_dist_m: float = MAX_DIST_M) -> pd.DataFrame:
    syn = stations_all[stations_all["source"] == "SYNOPTIC"].copy()
    syn = syn.dropna(subset=["latitude", "longitude", "source_station_id"]).reset_index(drop=True)

    matches = []
    for _, s in syn.iterrows():
        lat1, lon1 = float(s["latitude"]), float(s["longitude"])
        best = None
        best_d = None
        for _, r in itd_df.iterrows():
            d = haversine_m(lat1, lon1, float(r["latitude"]), float(r["longitude"]))
            if d <= max_dist_m and (best_d is None or d < best_d):
                best_d = d
                best = r
        if best is not None:
            matches.append({
                "SYNOPTIC_uid": f"SYNOPTIC:{s['source_station_id']}",
                "SYNOPTIC_stid": s["source_station_id"],
                "SYNOPTIC_name": s.get("station_name"),
                "SYNOPTIC_mnet_id": s.get("mnet_id") if "mnet_id" in s else s.get("network_id"),
                "ITD_site_id": best.get("itd_site_id"),
                "ITD_name": best.get("station_name"),
                "dist_m": float(best_d),
            })

    out = pd.DataFrame(matches).sort_values("dist_m").reset_index(drop=True)
    out["generated_utc"] = utcnow_iso()
    return out


# -------------------------
# MAIN
# -------------------------
def main():
    stations_all = pd.read_csv(STATIONS_ALL_CSV)

    # PATCH normalize network_id in stations
    if "network_id" in stations_all.columns:
        stations_all["network_id"] = (
            stations_all["network_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        )

    # 1) fetch networks
    networks_df = fetch_synoptic_networks(SYNOPTIC_TOKEN)

    # PATCH normalize networks_df and create mnet_id (BEFORE prints)
    if "ID" in networks_df.columns:
        networks_df["mnet_id"] = (
            networks_df["ID"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        )
    elif "id" in networks_df.columns:
        networks_df["mnet_id"] = (
            networks_df["id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        )
    else:
        raise KeyError(f"Cannot find network ID column in networks_df. Columns: {networks_df.columns.tolist()}")

    # DEBUG CHECK
    print("Unique network_id in stations:",
          stations_all[stations_all.source=="SYNOPTIC"]["network_id"].nunique())
    print("Unique mnet_id in networks table:",
          networks_df["mnet_id"].nunique())

    # Now merge/count
    networks_in_id = build_networks_in_idaho(stations_all, networks_df)
    print("Networks matched in Idaho:", len(networks_in_id))

    networks_in_id.to_csv(OUT_NETWORKS_IN_IDAHO, index=False)
    print(f"[OUT] {OUT_NETWORKS_IN_IDAHO} ({len(networks_in_id)} networks)")

    # 2) metadata completeness
    meta_comp = metadata_completeness_by_network(stations_all)
    meta_comp.to_csv(OUT_META_COMPLETENESS, index=False)
    print(f"[OUT] {OUT_META_COMPLETENESS} ({len(meta_comp)} networks)")

    # 3) overlap with ITD RWIS
    try:
        itd = fetch_itd_rwis_points()
        ov = overlap_synoptic_vs_itd(stations_all, itd, MAX_DIST_M)
        ov.to_csv(OUT_OVERLAP_ITD_RWIS, index=False)
        print(f"[OUT] {OUT_OVERLAP_ITD_RWIS} ({len(ov)} matches within {MAX_DIST_M} m)")
    except Exception as e:
        print(f"[WARN] ITD RWIS overlap skipped: {e}")

if __name__ == "__main__":
    main()

Unique network_id in stations: 41
Unique mnet_id in networks table: 398
Networks matched in Idaho: 41
[OUT] /Users/irene/projects/out/networks_in_idaho.csv (41 networks)
[OUT] /Users/irene/projects/out/metadata_completeness_by_network.csv (41 networks)
[OUT] /Users/irene/projects/out/overlap_synoptic_vs_ITD_Rwis.csv (119 matches within 200.0 m)


In [3]:
import pandas as pd

stations_all = pd.read_csv("/Users/irene/projects/out/stations_all.csv")
syn = stations_all[stations_all["source"]=="SYNOPTIC"].copy()

print("Syn rows:", len(syn))
print("Columns:", syn.columns.tolist())

for c in ["mnet_id","network_id","MNET_ID"]:
    if c in syn.columns:
        print(c, "nonnull:", syn[c].notna().sum(), "unique sample:", syn[c].dropna().astype(str).str.strip().unique()[:10])

Syn rows: 1175
Columns: ['source', 'source_station_id', 'station_name', 'latitude', 'longitude', 'elevation_m', 'county', 'huc', 'network_code', 'name_norm', 'station_uid', 'network_id', 'network_name']
network_id nonnull: 1175 unique sample: <StringArray>
['6.0', '1.0', '5.0', '11.0', '138.0', '25.0', '2.0', '41.0', '64.0', '65.0']
Length: 10, dtype: str


In [4]:
import pandas as pd

net = pd.read_csv("/Users/irene/projects/out/networks_in_idaho.csv")
meta = pd.read_csv("/Users/irene/projects/out/metadata_completeness_by_network.csv")
ov = pd.read_csv("/Users/irene/projects/out/overlap_synoptic_vs_ITD_Rwis.csv")

net.head(), meta.head(), ov.head()

(   mnet_id  n_stations  n_rows       SHORTNAME  \
 0        2         156     156            RAWS   
 1       41         129     129             ITD   
 2      204         116     116      USBR HYDRO   
 3     3022          95      95         TEMPEST   
 4       65          94      94  APRSWXNET/CWOP   
 
                                             LONGNAME                     URL  \
 0      Interagency Remote Automatic Weather Stations                     NaN   
 1                    Idaho Transportation Department                     NaN   
 2                           US Bureau of Reclamation                     NaN   
 3                                WeatherFlow Tempest  https://tempest.earth/   
 4  Automatic Position Reporting System WX NET/Cit...                     NaN   
 
    PROGRAM  CATEGORY              generated_utc  
 0      NaN         9  2026-02-16T22:54:16+00:00  
 1      NaN        10  2026-02-16T22:54:16+00:00  
 2      NaN         4  2026-02-16T22:54:16+00:00  


In [6]:
networks_df[networks_df["LONGNAME"].str.contains("Idaho", case=False, na=False)]

NameError: name 'networks_df' is not defined

In [5]:
net.sort_values("n_stations_idaho", ascending=False).head(15)

KeyError: 'n_stations_idaho'